
# Multi-Agent Reinforcement Learning (MARL) with TorchRL


In this assignment, you will implement and analyze two foundational algorithms in Multi-Agent Reinforcement Learning (MARL):
- [MADDPG (Multi-Agent Deep Deterministic Policy Gradient)](https://arxiv.org/abs/1706.02275$0)
- [MAPPO (Multi-Agent Proximal Policy Optimization)](https://arxiv.org/abs/2103.01955$0)

Both algorithms will be tested in the VMAS (Vectorized Multi-Agent Simulator) navigation environment, where multiple agents must learn to coordinate in a shared space.

In [ ]:
!pip3 install torchrl
!pip3 install vmas
!pip3 install tqdm
!apt-get update -y
!apt-get install -y x11-utils xvfb python3-opengl libgl1-mesa-glx libglu1-mesa
!pip install pyvirtualdisplay

In [ ]:
# Torch
import torch

# Tensordict modules
from tensordict import TensorDictBase
from tensordict.nn import set_composite_lp_aggregate, TensorDictModule, TensorDictSequential
from tensordict.nn.distributions import NormalParamExtractor
from torch import multiprocessing

# Data collection
from torchrl.collectors import Collector
from torchrl.data.replay_buffers import LazyMemmapStorage, RandomSampler, ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage

# Env
from torchrl.envs import RewardSum, TransformedEnv
from torchrl.envs.libs.vmas import VmasEnv
from torchrl.envs.utils import check_env_specs

# Multi-agent network
from torchrl.modules import MultiAgentMLP, ProbabilisticActor, AdditiveGaussianModule, TanhNormal, TanhDelta

# Loss
from torchrl.objectives import ClipPPOLoss, DDPGLoss, SoftUpdate, ValueEstimators

# Utils
torch.manual_seed(0)
from matplotlib import pyplot as plt
from tqdm import tqdm
import copy
import tempfile
from matplotlib import pyplot as plt


## Hyperparameters



In [ ]:
# Devices
is_fork = multiprocessing.get_start_method() == "fork"
device = (
    torch.device(0)
    if torch.cuda.is_available() and not is_fork
    else torch.device("cpu")
)
vmas_device = device  # The device where the simulator is run (VMAS can run on GPU)

# Sampling
frames_per_batch = 6_000
n_iters = 500  # Number of training iterations
total_frames = frames_per_batch * n_iters

# Training
num_epochs = 30  # Number of optimization steps per training iteration
minibatch_size = 400  # Size of the mini-batches in each optimization step
lr = 3e-4
max_grad_norm = 1.0  # Maximum norm for the gradients

gamma = 0.99  # discount factor

# DDPG Algorithm
polyak_tau = 0.005
memory_size=1_000_000

# PPO
clip_epsilon = 0.2  # clip value for PPO loss
lmbda = 0.9  # lambda for generalised advantage estimation
entropy_eps = 1e-4  # coefficient of the entropy term in the PPO loss

# disable log-prob aggregation
set_composite_lp_aggregate(False).set()

## Environment

We use the VMAS Navigation environment, a vectorized multi-agent simulator.
It runs many environments in parallel on the GPU, where each agent:
- observes its local state
- selects continuous actions
- receives individual rewards



In [ ]:
max_steps = 100
num_vmas_envs = (
    frames_per_batch // max_steps
)
scenario_name = "navigation"
n_agents = 3

env = VmasEnv(
    scenario=scenario_name,
    num_envs=num_vmas_envs,
    continuous_actions=True,
    max_steps=max_steps,
    device=vmas_device,
    n_agents=n_agents,
)

In [ ]:
print("action_spec:", env.full_action_spec)
print("reward_spec:", env.full_reward_spec)
print("done_spec:", env.full_done_spec)
print("observation_spec:", env.observation_spec)

In [ ]:
print("action_keys:", env.action_keys)
print("reward_keys:", env.reward_keys)
print("done_keys:", env.done_keys)

In [ ]:
env = TransformedEnv(
    env,
    RewardSum(in_keys=[env.reward_key], out_keys=[("agents", "episode_reward")]),
)

In [ ]:
check_env_specs(env)

In [ ]:
n_rollout_steps = 5
rollout = env.rollout(n_rollout_steps)
print("rollout of three steps:", rollout)
print("Shape of the rollout TensorDict:", rollout.batch_size)

# MADDPG

## Policy

In MADDPG, each agent learns a **deterministic policy** that maps its local observation directly to a continuous action. This actor is implemented as a shared MLP across agents, which promotes parameter sharing and improves sample efficiency by allowing agents to benefit from each other’s experience. Since actions must lie within bounded ranges, the raw network outputs are passed through a `tanh`-based transformation (e.g., `TanhDelta`) to ensure they stay within the valid action space.

Because the policy is deterministic, exploration is not inherent in the network and is instead introduced by adding Gaussian noise to the actions during training. This follows the MADDPG idea of combining stable deterministic policies with explicit exploration noise, enabling agents to explore the environment while still learning a smooth continuous control policy.

In [ ]:
share_parameters_policy = True

policy_net = MultiAgentMLP(
    n_agent_inputs=env.observation_spec["agents", "observation"].shape[
        -1
    ],  # n_obs_per_agent
    n_agent_outputs=env.full_action_spec["agents", "action"].shape[
        -1
    ],  # n_actions_per_agents
    n_agents=n_agents,
    centralised=False,  # the policies are decentralised
    share_params=share_parameters_policy,
    device=device,
    depth=2,
    num_cells=256,
    activation_class=torch.nn.Tanh,
)

In [ ]:
policy_module = TensorDictModule(
    policy_net,
    in_keys=[("agents", "observation")],
    out_keys=[("agents", "param")],
)

In [ ]:
### TODO: ###
# Create the actor using `ProbabilisticActor`.
# - Use the previously defined `policy_module`.
# - Use the environment's action specification (`env.full_action_spec["agents", "action"]`).
# - Read the policy output from ("agents", "param").
# - Write the sampled actions to ("agents", "action").
# - Use the `TanhDelta` distribution with the environment's action bounds.
# - Do not return the log probability of the actions.

policy = ProbabilisticActor(
    module=policy_module,
    spec=env.full_action_spec["agents", "action"],
    in_keys=[("agents", "param")],
    out_keys=[("agents", "action")],
    distribution_class=TanhDelta,
    distribution_kwargs={
        "low": env.full_action_spec_unbatched["agents", "action"].space.low,
        "high": env.full_action_spec_unbatched["agents", "action"].space.high,
    },
    return_log_prob=False,
)


In [ ]:
exploration_policy = TensorDictSequential(
        policy,
        AdditiveGaussianModule(
            spec=policy.spec,
            annealing_num_steps=total_frames
            // 2,  # Number of frames after which sigma is sigma_end
            action_key=env.action_key,
            sigma_init=0.5,  # Initial value of the sigma
            sigma_end=0.5,  # Final value of the sigma
        ),
)

## Critic network
In MADDPG, the critic is **centralized during training**, meaning it receives the concatenated observations and actions of all agents. This allows it to estimate a joint action-value function \( Q(s, a_1, ..., a_n) \), capturing interactions between agents that are not visible locally.

This centralized design improves training stability by reducing non-stationarity, while agents still execute policies **decentrally using only local observations**, following the CTDE paradigm.

In [ ]:
share_parameters_critic = True
MADDPG = True  # IDDPG if False

### TODO: ###
# Create a TensorDictModule that concatenates each agent's observation and action.
# - Read the observation from ("agents", "observation").
# - Read the action from ("agents", "action").
# - Concatenate them along the last dimension using `torch.cat`.
# - Store the result in ("agents", "obs_action").
cat_module = TensorDictModule(
    lambda obs, action: torch.cat([obs, action], dim=-1),
    in_keys=[("agents", "observation"), ("agents", "action")],
    out_keys=[("agents", "obs_action")],
)

critic_module = TensorDictModule(
    module=MultiAgentMLP(
        n_agent_inputs=env.observation_spec["agents", "observation"].shape[-1]
        + env.full_action_spec["agents", "action"].shape[-1],
        n_agent_outputs=1,
        n_agents=n_agents,
        centralised=MADDPG,
        share_params=share_parameters_critic,
        device=device,
        depth=2,
        num_cells=256,
        activation_class=torch.nn.Tanh,
    ),
    in_keys=[("agents", "obs_action")],
    out_keys=[
        ("agents", "state_action_value")
    ],
)

critic = TensorDictSequential(
    cat_module, critic_module
)

# CTDE shape check: the actor receives one local observation per agent,
# while a centralized critic internally uses all agents' obs-action features.
with torch.no_grad():
    _shape_td = env.reset()
    _actor_td = policy(_shape_td.clone())
    _critic_td = critic(_actor_td.clone())

_local_obs_dim = _shape_td.get(("agents", "observation")).shape[-1]
_local_action_dim = _actor_td.get(("agents", "action")).shape[-1]
_critic_local_feature_dim = _local_obs_dim + _local_action_dim
_critic_effective_input_dim = (
    n_agents * _critic_local_feature_dim if MADDPG else _critic_local_feature_dim
)

print("Actor observation tensor shape:", _shape_td.get(("agents", "observation")).shape)
print("Actor local input width:", _local_obs_dim)
print("Critic obs-action tensor shape:", _critic_td.get(("agents", "obs_action")).shape)
print("Critic effective input width:", _critic_effective_input_dim)
print("Critic output shape:", _critic_td.get(("agents", "state_action_value")).shape)


## Data collector

In [ ]:
collector = Collector(
    env,
    exploration_policy,
    device=device,
    frames_per_batch=frames_per_batch,
    total_frames=total_frames,
)

## Replay buffer

In [ ]:
scratch_dir = tempfile.TemporaryDirectory().name
replay_buffer = ReplayBuffer(
    storage=LazyMemmapStorage(
        memory_size,
        scratch_dir=scratch_dir,
    ),  # store up to memory_size multi-agent transitions
    sampler=RandomSampler(),
    batch_size=minibatch_size,  # sample batches of this size
)
if device.type != "cpu":
    replay_buffer.append_transform(lambda x: x.to(device))

## Loss function

The DDPG loss consists of:
- Actor loss: improves policy using critic feedback
- Critic loss: fits Q-values using TD targets

Target networks and soft updates stabilize training.

In [ ]:
loss_module = DDPGLoss(
    actor_network=policy,  # Use the non-explorative policies
    value_network=critic,
    delay_value=True,  # Whether to use a target network for the value
    loss_function="l2",
)
loss_module.set_keys(
    state_action_value=("agents", "state_action_value"),
    reward=("agents", "reward"),
    done=("agents", "done"),
    terminated=("agents", "terminated"),
)
loss_module.make_value_estimator(ValueEstimators.TD0, gamma=gamma)

In [ ]:
target_updater = SoftUpdate(loss_module, tau=polyak_tau)

In [ ]:
optimisers = {
        "loss_actor": torch.optim.Adam(
            loss_module.actor_network_params.flatten_keys().values(), lr=lr
        ),
        "loss_value": torch.optim.Adam(
            loss_module.value_network_params.flatten_keys().values(), lr=lr
        ),
    }

## Training Loop

In [ ]:
reward_key = env.reward_key
done_key = ("next", "done")
terminated_key = ("next", "terminated")

pbar = tqdm(total=n_iters, desc="Training MADDPG...")
episode_reward_mean_list = []

maddpg_actor_loss_list = []
maddpg_critic_loss_list = []
maddpg_grad_norm_list = []
maddpg_exploration_scale_list = []
maddpg_env_frames_list = []
collected_frames = 0

for tensordict_data in collector:
    current_frames = tensordict_data.numel()
    collected_frames += current_frames

    reward_shape = tensordict_data.get_item_shape(("next", reward_key))

    tensordict_data.set(
        ("next", "agents", "done"),
        tensordict_data.get(done_key).unsqueeze(-1).expand(reward_shape),
    )
    tensordict_data.set(
        ("next", "agents", "terminated"),
        tensordict_data.get(terminated_key).unsqueeze(-1).expand(reward_shape),
    )

    # Add to Replay Buffer
    data_view = tensordict_data.reshape(-1)
    replay_buffer.extend(data_view)

    last_actor_loss = float("nan")
    last_critic_loss = float("nan")
    last_grad_norm = float("nan")

    if len(replay_buffer) >= minibatch_size:
        for _ in range(num_epochs):
            ### TODO:  ###
            # Complete one MADDPG optimization step.
            # - Sample a minibatch of transitions from the replay buffer.
            # - Compute the actor and critic losses using `loss_module`.
            # - Compute the total loss by summing the actor and critic losses.
            # - Backpropagate the gradients.
            # - Clip the gradients using `torch.nn.utils.clip_grad_norm_`.
            # - Update both the actor and critic optimizers.
            # - Clear the gradients of both optimizers.
            # - Soft-update the target networks.
            subdata = replay_buffer.sample()
            loss_vals = loss_module(subdata)

            actor_loss = loss_vals["loss_actor"]
            critic_loss = loss_vals["loss_value"]
            total_loss = actor_loss + critic_loss

            optimisers["loss_actor"].zero_grad(set_to_none=True)
            optimisers["loss_value"].zero_grad(set_to_none=True)

            total_loss.backward()

            actor_params = optimisers["loss_actor"].param_groups[0]["params"]
            critic_params = optimisers["loss_value"].param_groups[0]["params"]
            actor_grad_norm = torch.nn.utils.clip_grad_norm_(
                actor_params, max_grad_norm
            )
            critic_grad_norm = torch.nn.utils.clip_grad_norm_(
                critic_params, max_grad_norm
            )

            optimisers["loss_actor"].step()
            optimisers["loss_value"].step()

            optimisers["loss_actor"].zero_grad(set_to_none=True)
            optimisers["loss_value"].zero_grad(set_to_none=True)

            target_updater.step()

            last_actor_loss = actor_loss.detach().item()
            last_critic_loss = critic_loss.detach().item()
            last_grad_norm = max(
                float(actor_grad_norm.detach().cpu()),
                float(critic_grad_norm.detach().cpu()),
            )

    exploration_policy[-1].step(current_frames)

    maddpg_actor_loss_list.append(last_actor_loss)
    maddpg_critic_loss_list.append(last_critic_loss)
    maddpg_grad_norm_list.append(last_grad_norm)
    maddpg_exploration_scale_list.append(
        float(exploration_policy[-1].sigma.detach().cpu())
    )
    maddpg_env_frames_list.append(collected_frames)

    done = tensordict_data.get(("next", "agents", "done"))
    if done.any():
        episode_reward_mean = (
            tensordict_data.get(("next", "agents", "episode_reward"))[done].mean().item()
        )
        episode_reward_mean_list.append(episode_reward_mean)
        pbar.set_description(
            (
                f"return={episode_reward_mean:.2f}, "
                f"actor={last_actor_loss:.3f}, critic={last_critic_loss:.3f}, "
                f"sigma={maddpg_exploration_scale_list[-1]:.3f}, "
                f"frames={collected_frames}"
            ),
            refresh=False,
        )

    pbar.update()


## Results

In [ ]:
plt.plot(episode_reward_mean_list)
plt.xlabel("Training iterations")
plt.ylabel("Reward")
plt.title("Episode reward mean")
plt.show()

## Render

In [ ]:
import pyvirtualdisplay
display = pyvirtualdisplay.Display(visible=False, size=(1400, 900))
display.start()
from PIL import Image

def rendering_callback(env, td):
    env.frames.append(Image.fromarray(env.render(mode="rgb_array")))
env.frames = []
with torch.no_grad():
   env.rollout(
       max_steps=max_steps,
       policy=policy,
       callback=rendering_callback,
       auto_cast_to_device=True,
       break_when_any_done=False,
   )
env.frames[0].save(
    f"{scenario_name}.gif",
    save_all=True,
    append_images=env.frames[1:],
   duration=3,
   loop=0,
)

from IPython.display import Image
Image(open(f"{scenario_name}.gif", "rb").read())

# MAPPO

## Hyperparameters


In [ ]:
frames_per_batch = 6_000
n_iters = 20
total_frames = frames_per_batch * n_iters

## Environment

In [ ]:
max_steps = 100
num_vmas_envs = (
    frames_per_batch // max_steps
)
scenario_name = "navigation"
n_agents = 3

env = VmasEnv(
    scenario=scenario_name,
    num_envs=num_vmas_envs,
    continuous_actions=True,
    max_steps=max_steps,
    device=vmas_device,
    n_agents=n_agents,
)


In [ ]:
env = TransformedEnv(
    env,
    RewardSum(in_keys=[env.reward_key], out_keys=[("agents", "episode_reward")]),
)

## Policy
In MAPPO, the policy is stochastic and outputs the parameters of a Gaussian distribution (mean and standard deviation), from which actions are sampled using a squashed distribution like TanhNormal to ensure valid action bounds.

In [ ]:
share_parameters_policy = True

policy_net = torch.nn.Sequential(
    MultiAgentMLP(
        n_agent_inputs=env.observation_spec["agents", "observation"].shape[
            -1
        ],  # n_obs_per_agent
        n_agent_outputs=2
        * env.full_action_spec[env.action_key].shape[-1],  # 2 * n_actions_per_agents
        n_agents=env.n_agents,
        centralised=False,  # the policies are decentralised
        share_params=share_parameters_policy,
        device=device,
        depth=2,
        num_cells=256,
        activation_class=torch.nn.Tanh,
    ),
    NormalParamExtractor(),  # this will just separate the last dimension into two outputs: a loc and a non-negative scale
)

In [ ]:
policy_module = TensorDictModule(
    policy_net,
    in_keys=[("agents", "observation")],
    out_keys=[("agents", "loc"), ("agents", "scale")],
)

In [ ]:
### TODO:  ###
# Create the stochastic actor using `ProbabilisticActor`.
# - Use the previously defined `policy_module`.
# - Use the environment's unbatched action specification (`env.action_spec_unbatched`).
# - Read the Gaussian mean and standard deviation from ("agents", "loc") and ("agents", "scale").
# - Write the sampled actions to `env.action_key`.
# - Use the `TanhNormal` distribution with the environment's action bounds.
# - Return the log probability of the sampled actions for PPO training.

policy = ProbabilisticActor(
    module=policy_module,
    spec=env.action_spec_unbatched,
    in_keys=[("agents", "loc"), ("agents", "scale")],
    out_keys=[env.action_key],
    distribution_class=TanhNormal,
    distribution_kwargs={
        "low": env.full_action_spec_unbatched[env.action_key].space.low,
        "high": env.full_action_spec_unbatched[env.action_key].space.high,
    },
    return_log_prob=True,
)


## Critic network
In MAPPO, the critic estimates the state-value function for each agent using centralized observations. It is used to compute advantage estimates via GAE (Generalized Advantage Estimation), which helps reduce variance and stabilize policy updates during PPO training.


In [ ]:
share_parameters_critic = True
mappo = True  # IPPO if False

critic_net = MultiAgentMLP(
    n_agent_inputs=env.observation_spec["agents", "observation"].shape[-1],
    n_agent_outputs=1,  # 1 value per agent
    n_agents=env.n_agents,
    centralised=mappo,
    share_params=share_parameters_critic,
    device=device,
    depth=2,
    num_cells=256,
    activation_class=torch.nn.Tanh,
)

critic = TensorDictModule(
    module=critic_net,
    in_keys=[("agents", "observation")],
    out_keys=[("agents", "state_value")],
)

In [ ]:
print("Running policy:", policy(env.reset()))
print("Running value:", critic(env.reset()))

## Data collector




In [ ]:
collector = Collector(
    env,
    policy,
    device=vmas_device,
    storing_device=device,
    frames_per_batch=frames_per_batch,
    total_frames=total_frames,
)

## Replay buffer



In [ ]:
replay_buffer = ReplayBuffer(
    storage=LazyTensorStorage(
        frames_per_batch, device=device
    ),  # store the frames_per_batch collected at each iteration
    sampler=SamplerWithoutReplacement(),
    batch_size=minibatch_size,  # sample minibatches of this size
)

## Loss function

PPO optimizes a clipped objective composed of:
- policy loss (improves actions while staying close to old policy)
- value loss (critic accuracy)
- entropy bonus (encourages exploration)

Advantage normalization is disabled across agents to preserve multi-agent structure.



In [ ]:
### TODO: ###
# Create the PPO loss module using `ClipPPOLoss`.
# - Use the MAPPO actor network (`policy`) and critic network (`critic`).
# - Set `clip_epsilon` to control policy update clipping.
# - Set `entropy_coeff` to encourage exploration.
# - Ensure advantage normalization is disabled (`normalize_advantage=False`)
#   to avoid mixing statistics across multiple agents.

loss_module = ClipPPOLoss(
    actor_network=policy,
    critic_network=critic,
    clip_epsilon=clip_epsilon,
    entropy_coeff=entropy_eps,
    normalize_advantage=False,
)

loss_module.set_keys(
    reward=env.reward_key,
    action=env.action_key,
    value=("agents", "state_value"),
    done=("agents", "done"),
    terminated=("agents", "terminated"),
)


loss_module.make_value_estimator(
    ValueEstimators.GAE, gamma=gamma, lmbda=lmbda
)
GAE = loss_module.value_estimator

optim = torch.optim.Adam(loss_module.parameters(), lr)


## Training loop






In [ ]:
pbar = tqdm(total=n_iters, desc="episode_reward_mean = 0")

episode_reward_mean_list = []

# Diagnostics requested by the assignment.
mappo_policy_loss_list = []
mappo_critic_loss_list = []
mappo_entropy_loss_list = []
mappo_kl_list = []
mappo_ratio_stat_list = []
mappo_grad_norm_list = []
mappo_env_frames_list = []
collected_frames = 0
_shape_check_done = False

for tensordict_data in collector:
    current_frames = tensordict_data.numel()
    collected_frames += current_frames

    tensordict_data.set(
        ("next", "agents", "done"),
        tensordict_data.get(("next", "done"))
        .unsqueeze(-1)
        .expand(tensordict_data.get_item_shape(("next", env.reward_key))),
    )
    tensordict_data.set(
        ("next", "agents", "terminated"),
        tensordict_data.get(("next", "terminated"))
        .unsqueeze(-1)
        .expand(tensordict_data.get_item_shape(("next", env.reward_key))),
    )


    with torch.no_grad():
        GAE(
            tensordict_data,
            params=loss_module.critic_network_params,
            target_params=loss_module.target_critic_network_params,
        )

    if not _shape_check_done:
        with torch.no_grad():
            _value_check = critic(tensordict_data.clone()).get(
                ("agents", "state_value")
            )
        print("MAPPO value_target/return shape:", tensordict_data.get("value_target").shape)
        print("MAPPO value shape:", _value_check.shape)
        print("MAPPO advantage shape:", tensordict_data.get("advantage").shape)
        print(
            "MAPPO action log-prob shape:",
            tensordict_data.get(("agents", "action_log_prob")).shape,
        )
        _shape_check_done = True

    data_view = tensordict_data.reshape(-1)  # Flatten the batch size to shuffle data
    replay_buffer.extend(data_view)

    last_policy_loss = float("nan")
    last_critic_loss = float("nan")
    last_entropy_loss = float("nan")
    last_kl = float("nan")
    last_ratio_stat = float("nan")
    last_grad_norm = float("nan")

    for _ in range(num_epochs):
        for _ in range(frames_per_batch // minibatch_size):
            ### TODO: ###
            # Complete one PPO optimization step.
            # - Sample a minibatch from the replay buffer.
            # - Compute PPO loss components using `loss_module`:
            #   * policy loss (loss_objective)
            #   * critic loss (loss_critic)
            #   * entropy bonus (loss_entropy)
            # - Combine all loss terms into a single scalar loss.
            # - Perform backpropagation.
            # - Apply gradient clipping using `clip_grad_norm_`.
            # - Perform an optimizer step.
            # - Reset gradients after the update.
            subdata = replay_buffer.sample()
            loss_vals = loss_module(subdata)

            policy_loss = loss_vals["loss_objective"]
            critic_loss = loss_vals["loss_critic"]
            entropy_loss = loss_vals["loss_entropy"]
            loss_value = policy_loss + critic_loss + entropy_loss

            optim.zero_grad(set_to_none=True)
            loss_value.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(
                loss_module.parameters(), max_grad_norm
            )
            optim.step()
            optim.zero_grad(set_to_none=True)

            last_policy_loss = policy_loss.detach().item()
            last_critic_loss = critic_loss.detach().item()
            last_entropy_loss = entropy_loss.detach().item()
            last_grad_norm = float(grad_norm.detach().cpu())

            loss_keys = set(loss_vals.keys())
            if "kl_approx" in loss_keys:
                last_kl = loss_vals["kl_approx"].detach().mean().item()
            if "clip_fraction" in loss_keys:
                last_ratio_stat = loss_vals["clip_fraction"].detach().mean().item()
            elif "ESS" in loss_keys:
                last_ratio_stat = loss_vals["ESS"].detach().mean().item()

    collector.update_policy_weights_()

    mappo_policy_loss_list.append(last_policy_loss)
    mappo_critic_loss_list.append(last_critic_loss)
    mappo_entropy_loss_list.append(last_entropy_loss)
    mappo_kl_list.append(last_kl)
    mappo_ratio_stat_list.append(last_ratio_stat)
    mappo_grad_norm_list.append(last_grad_norm)
    mappo_env_frames_list.append(collected_frames)

    # Logging
    done = tensordict_data.get(("next", "agents", "done"))
    episode_reward_mean = (
        tensordict_data.get(("next", "agents", "episode_reward"))[done].mean().item()
    )
    episode_reward_mean_list.append(episode_reward_mean)
    pbar.set_description(
        (
            f"return={episode_reward_mean:.2f}, "
            f"policy={last_policy_loss:.3f}, critic={last_critic_loss:.3f}, "
            f"entropy={last_entropy_loss:.3f}, kl={last_kl:.4f}, "
            f"ratio_stat={last_ratio_stat:.4f}, frames={collected_frames}"
        ),
        refresh=False,
    )
    pbar.update()


## Results




In [ ]:
plt.plot(episode_reward_mean_list)
plt.xlabel("Training iterations")
plt.ylabel("Reward")
plt.title("Episode reward mean")
plt.show()

## Render

In [ ]:
import pyvirtualdisplay
display = pyvirtualdisplay.Display(visible=False, size=(1400, 900))
display.start()
from PIL import Image

def rendering_callback(env, td):
    env.frames.append(Image.fromarray(env.render(mode="rgb_array")))
env.frames = []
with torch.no_grad():
   env.rollout(
       max_steps=max_steps,
       policy=policy,
       callback=rendering_callback,
       auto_cast_to_device=True,
       break_when_any_done=False,
   )
env.frames[0].save(
    f"{scenario_name}.gif",
    save_all=True,
    append_images=env.frames[1:],
   duration=3,
   loop=0,
)

from IPython.display import Image
Image(open(f"{scenario_name}.gif", "rb").read())

# Controlled CTDE Comparison (Implementation 6)

This section runs a controlled ablation of **MADDPG, IDDPG, MAPPO, and IPPO** from scratch.

- The VMAS `navigation` task, number of agents, episode length, network depth/width, optimizer settings, and **environment-frame budget are held fixed**.
- `MADDPG -> IDDPG` changes only the DDPG critic from centralized to decentralized.
- `MAPPO -> IPPO` changes only the PPO critic from centralized to decentralized.
- Actors stay decentralized and parameter-shared in all four methods.
- Every method is run with multiple seeds and plotted against **environment frames**, not training iterations.
- Final evaluation uses the actor **without DDPG Gaussian exploration noise** and deterministic action selection for PPO.


In [ ]:
# Implementation 6: controlled CTDE comparison
import gc
import random
import numpy as np
import pandas as pd
from IPython.display import display as ipy_display
from torchrl.envs import ExplorationType, set_exploration_type

# A practical, equal interaction budget for all four methods.
# These defaults give 3 independent seeds as required by the assignment.
comparison_seeds = [0, 1, 2]
comparison_frames_per_batch = 6_000
comparison_total_frames = 120_000
comparison_n_iters = comparison_total_frames // comparison_frames_per_batch
comparison_max_steps = 100
comparison_num_envs = comparison_frames_per_batch // comparison_max_steps
comparison_minibatch_size = minibatch_size

# Preserve the optimization style used above while keeping the *environment*
# interaction budget identical across algorithms.
comparison_ddpg_updates_per_batch = num_epochs
comparison_ppo_epochs = num_epochs
comparison_memory_size = comparison_total_frames

comparison_algorithms = ["MADDPG", "IDDPG", "MAPPO", "IPPO"]

print("Controlled-comparison configuration")
print("-----------------------------------")
print(f"Algorithms: {comparison_algorithms}")
print(f"Seeds: {comparison_seeds}")
print(f"Training frames per algorithm/seed: {comparison_total_frames:,}")
print(f"Frames per collected batch: {comparison_frames_per_batch:,}")
print(f"VMAS envs: {comparison_num_envs}, episode length: {comparison_max_steps}")
print(
    "DDPG gradient updates per seed: "
    f"{comparison_n_iters * comparison_ddpg_updates_per_batch:,}"
)
print(
    "PPO minibatch updates per seed: "
    f"{comparison_n_iters * comparison_ppo_epochs * (comparison_frames_per_batch // comparison_minibatch_size):,}"
)
print(
    "Total training interactions across all runs: "
    f"{len(comparison_algorithms) * len(comparison_seeds) * comparison_total_frames:,} frames"
)


def _set_comparison_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _make_comparison_env(seed, num_envs=None):
    if num_envs is None:
        num_envs = comparison_num_envs

    comparison_env = VmasEnv(
        scenario=scenario_name,
        num_envs=num_envs,
        continuous_actions=True,
        max_steps=comparison_max_steps,
        device=vmas_device,
        n_agents=n_agents,
    )
    comparison_env = TransformedEnv(
        comparison_env,
        RewardSum(
            in_keys=[comparison_env.reward_key],
            out_keys=[("agents", "episode_reward")],
        ),
    )
    comparison_env.set_seed(seed)
    return comparison_env


def _build_ddpg_actor(comparison_env):
    actor_net = MultiAgentMLP(
        n_agent_inputs=comparison_env.observation_spec["agents", "observation"].shape[-1],
        n_agent_outputs=comparison_env.full_action_spec["agents", "action"].shape[-1],
        n_agents=n_agents,
        centralised=False,
        share_params=True,
        device=device,
        depth=2,
        num_cells=256,
        activation_class=torch.nn.Tanh,
    )

    actor_module = TensorDictModule(
        actor_net,
        in_keys=[("agents", "observation")],
        out_keys=[("agents", "param")],
    )

    return ProbabilisticActor(
        module=actor_module,
        spec=comparison_env.full_action_spec["agents", "action"],
        in_keys=[("agents", "param")],
        out_keys=[("agents", "action")],
        distribution_class=TanhDelta,
        distribution_kwargs={
            "low": comparison_env.full_action_spec_unbatched[
                "agents", "action"
            ].space.low,
            "high": comparison_env.full_action_spec_unbatched[
                "agents", "action"
            ].space.high,
        },
        return_log_prob=False,
    )


def _build_ddpg_critic(comparison_env, centralised):
    cat_module = TensorDictModule(
        lambda obs, action: torch.cat([obs, action], dim=-1),
        in_keys=[("agents", "observation"), ("agents", "action")],
        out_keys=[("agents", "obs_action")],
    )

    critic_module = TensorDictModule(
        module=MultiAgentMLP(
            n_agent_inputs=(
                comparison_env.observation_spec["agents", "observation"].shape[-1]
                + comparison_env.full_action_spec["agents", "action"].shape[-1]
            ),
            n_agent_outputs=1,
            n_agents=n_agents,
            centralised=centralised,
            share_params=True,
            device=device,
            depth=2,
            num_cells=256,
            activation_class=torch.nn.Tanh,
        ),
        in_keys=[("agents", "obs_action")],
        out_keys=[("agents", "state_action_value")],
    )

    return TensorDictSequential(cat_module, critic_module)


def _build_ppo_actor(comparison_env):
    actor_net = torch.nn.Sequential(
        MultiAgentMLP(
            n_agent_inputs=comparison_env.observation_spec["agents", "observation"].shape[-1],
            n_agent_outputs=2
            * comparison_env.full_action_spec[comparison_env.action_key].shape[-1],
            n_agents=n_agents,
            centralised=False,
            share_params=True,
            device=device,
            depth=2,
            num_cells=256,
            activation_class=torch.nn.Tanh,
        ),
        NormalParamExtractor(),
    )

    actor_module = TensorDictModule(
        actor_net,
        in_keys=[("agents", "observation")],
        out_keys=[("agents", "loc"), ("agents", "scale")],
    )

    return ProbabilisticActor(
        module=actor_module,
        spec=comparison_env.action_spec_unbatched,
        in_keys=[("agents", "loc"), ("agents", "scale")],
        out_keys=[comparison_env.action_key],
        distribution_class=TanhNormal,
        distribution_kwargs={
            "low": comparison_env.full_action_spec_unbatched[
                comparison_env.action_key
            ].space.low,
            "high": comparison_env.full_action_spec_unbatched[
                comparison_env.action_key
            ].space.high,
        },
        return_log_prob=True,
    )


def _build_ppo_critic(comparison_env, centralised):
    critic_net = MultiAgentMLP(
        n_agent_inputs=comparison_env.observation_spec["agents", "observation"].shape[-1],
        n_agent_outputs=1,
        n_agents=n_agents,
        centralised=centralised,
        share_params=True,
        device=device,
        depth=2,
        num_cells=256,
        activation_class=torch.nn.Tanh,
    )

    return TensorDictModule(
        module=critic_net,
        in_keys=[("agents", "observation")],
        out_keys=[("agents", "state_value")],
    )


def _expand_agent_done_flags(tensordict_data, reward_key):
    reward_shape = tensordict_data.get_item_shape(("next", reward_key))
    tensordict_data.set(
        ("next", "agents", "done"),
        tensordict_data.get(("next", "done")).unsqueeze(-1).expand(reward_shape),
    )
    tensordict_data.set(
        ("next", "agents", "terminated"),
        tensordict_data.get(("next", "terminated")).unsqueeze(-1).expand(reward_shape),
    )


def _completed_per_agent_returns(tensordict_data):
    """Return completed episode returns with shape [episodes, n_agents]."""
    done_env = tensordict_data.get(("next", "done")).squeeze(-1)
    episode_return = tensordict_data.get(
        ("next", "agents", "episode_reward")
    ).squeeze(-1)

    completed = episode_return[done_env]
    if completed.numel() == 0:
        # Fallback for an evaluation rollout that reaches max_steps without
        # exposing a done entry in the returned TensorDict.
        completed = episode_return[:, -1]

    if completed.ndim == 1:
        completed = completed.unsqueeze(0)
    return completed.reshape(-1, n_agents)


def _batch_team_return(tensordict_data):
    completed = _completed_per_agent_returns(tensordict_data)
    if completed.numel() == 0:
        return float("nan")
    return completed.sum(dim=-1).mean().item()


def _cpu_state_dict(module):
    return {
        key: value.detach().cpu().clone()
        for key, value in module.state_dict().items()
    }


def _evaluate_final_policy(policy, family, seed):
    # Use the same vectorization as training so the actor spec has matching
    # batch dimensions. Evaluation frames are not counted in the training budget.
    eval_env = _make_comparison_env(seed + 10_000, num_envs=comparison_num_envs)
    policy.eval()

    with torch.no_grad(), set_exploration_type(ExplorationType.DETERMINISTIC):
        rollout = eval_env.rollout(
            max_steps=comparison_max_steps,
            policy=policy,
            auto_cast_to_device=True,
            break_when_any_done=False,
        )

    completed = _completed_per_agent_returns(rollout)
    per_agent_mean = completed.mean(dim=0)
    team_returns = completed.sum(dim=-1)

    metrics = {
        "team_return_mean": team_returns.mean().item(),
        "team_return_std": team_returns.std(unbiased=False).item(),
        "per_agent_return": per_agent_mean.detach().cpu().tolist(),
        "n_eval_episodes": int(completed.shape[0]),
    }

    try:
        eval_env.close()
    except Exception:
        pass

    return metrics


In [ ]:
def _train_ddpg_comparison(seed, centralised, save_policy=False):
    algorithm = "MADDPG" if centralised else "IDDPG"
    _set_comparison_seed(seed)

    comparison_env = _make_comparison_env(seed)
    actor = _build_ddpg_actor(comparison_env)
    critic = _build_ddpg_critic(comparison_env, centralised=centralised)

    # Gaussian noise exists only in the collector policy. The DDPG loss sees
    # the deterministic base actor above.
    collector_policy = TensorDictSequential(
        actor,
        AdditiveGaussianModule(
            spec=actor.spec,
            annealing_num_steps=comparison_total_frames // 2,
            action_key=comparison_env.action_key,
            sigma_init=0.5,
            sigma_end=0.5,
        ),
    )

    collector_cmp = Collector(
        comparison_env,
        collector_policy,
        device=device,
        frames_per_batch=comparison_frames_per_batch,
        total_frames=comparison_total_frames,
        auto_register_policy_transforms=True,
    )

    replay_cmp = ReplayBuffer(
        storage=LazyTensorStorage(comparison_memory_size, device=device),
        sampler=RandomSampler(),
        batch_size=comparison_minibatch_size,
    )

    ddpg_loss = DDPGLoss(
        actor_network=actor,
        value_network=critic,
        delay_value=True,
        loss_function="l2",
    )
    ddpg_loss.set_keys(
        state_action_value=("agents", "state_action_value"),
        reward=("agents", "reward"),
        done=("agents", "done"),
        terminated=("agents", "terminated"),
    )
    ddpg_loss.make_value_estimator(ValueEstimators.TD0, gamma=gamma)
    ddpg_target_updater = SoftUpdate(ddpg_loss, tau=polyak_tau)

    actor_optim = torch.optim.Adam(
        ddpg_loss.actor_network_params.flatten_keys().values(), lr=lr
    )
    critic_optim = torch.optim.Adam(
        ddpg_loss.value_network_params.flatten_keys().values(), lr=lr
    )

    frames_log = []
    return_log = []
    critic_loss_log = []
    grad_norm_log = []
    exploration_log = []
    collected_frames = 0

    pbar_cmp = tqdm(
        total=comparison_n_iters,
        desc=f"{algorithm} seed={seed}",
        leave=False,
    )

    for batch in collector_cmp:
        current_frames = batch.numel()
        collected_frames += current_frames

        _expand_agent_done_flags(batch, comparison_env.reward_key)
        replay_cmp.extend(batch.reshape(-1))

        last_critic_loss = float("nan")
        last_grad_norm = float("nan")

        if len(replay_cmp) >= comparison_minibatch_size:
            for _ in range(comparison_ddpg_updates_per_batch):
                subdata = replay_cmp.sample()
                loss_vals = ddpg_loss(subdata)

                actor_loss = loss_vals["loss_actor"]
                critic_loss = loss_vals["loss_value"]
                total_loss = actor_loss + critic_loss

                actor_optim.zero_grad(set_to_none=True)
                critic_optim.zero_grad(set_to_none=True)
                total_loss.backward()

                actor_grad = torch.nn.utils.clip_grad_norm_(
                    actor_optim.param_groups[0]["params"], max_grad_norm
                )
                critic_grad = torch.nn.utils.clip_grad_norm_(
                    critic_optim.param_groups[0]["params"], max_grad_norm
                )

                actor_optim.step()
                critic_optim.step()
                actor_optim.zero_grad(set_to_none=True)
                critic_optim.zero_grad(set_to_none=True)
                ddpg_target_updater.step()

                last_critic_loss = critic_loss.detach().item()
                last_grad_norm = max(
                    float(actor_grad.detach().cpu()),
                    float(critic_grad.detach().cpu()),
                )

        collector_policy[-1].step(current_frames)
        collector_cmp.update_policy_weights_()

        batch_return = _batch_team_return(batch)
        frames_log.append(collected_frames)
        return_log.append(batch_return)
        critic_loss_log.append(last_critic_loss)
        grad_norm_log.append(last_grad_norm)
        exploration_log.append(
            float(collector_policy[-1].sigma.detach().cpu())
        )

        pbar_cmp.set_postfix(
            frames=collected_frames,
            team_return=f"{batch_return:.2f}",
            refresh=False,
        )
        pbar_cmp.update()

    pbar_cmp.close()

    # Critically, evaluate the base actor, not collector_policy, so there is
    # no Gaussian exploration noise at evaluation time.
    evaluation = _evaluate_final_policy(actor, "DDPG", seed)
    saved_state = _cpu_state_dict(actor) if save_policy else None

    result = {
        "algorithm": algorithm,
        "seed": seed,
        "frames": frames_log,
        "team_return": return_log,
        "critic_loss": critic_loss_log,
        "grad_norm": grad_norm_log,
        "exploration_scale": exploration_log,
        "evaluation": evaluation,
    }

    try:
        collector_cmp.shutdown()
    except Exception:
        pass
    try:
        comparison_env.close()
    except Exception:
        pass

    del replay_cmp, ddpg_loss, critic, collector_policy, actor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result, saved_state


In [ ]:
def _train_ppo_comparison(seed, centralised, save_policy=False):
    algorithm = "MAPPO" if centralised else "IPPO"
    _set_comparison_seed(seed)

    comparison_env = _make_comparison_env(seed)
    actor = _build_ppo_actor(comparison_env)
    critic = _build_ppo_critic(comparison_env, centralised=centralised)

    collector_cmp = Collector(
        comparison_env,
        actor,
        device=vmas_device,
        storing_device=device,
        frames_per_batch=comparison_frames_per_batch,
        total_frames=comparison_total_frames,
        auto_register_policy_transforms=True,
    )

    replay_cmp = ReplayBuffer(
        storage=LazyTensorStorage(comparison_frames_per_batch, device=device),
        sampler=SamplerWithoutReplacement(),
        batch_size=comparison_minibatch_size,
    )

    ppo_loss = ClipPPOLoss(
        actor_network=actor,
        critic_network=critic,
        clip_epsilon=clip_epsilon,
        entropy_coeff=entropy_eps,
        normalize_advantage=False,
    )
    ppo_loss.set_keys(
        reward=comparison_env.reward_key,
        action=comparison_env.action_key,
        value=("agents", "state_value"),
        done=("agents", "done"),
        terminated=("agents", "terminated"),
    )
    ppo_loss.make_value_estimator(ValueEstimators.GAE, gamma=gamma, lmbda=lmbda)
    gae_cmp = ppo_loss.value_estimator
    ppo_optim = torch.optim.Adam(ppo_loss.parameters(), lr)

    frames_log = []
    return_log = []
    critic_loss_log = []
    grad_norm_log = []
    entropy_loss_log = []
    kl_log = []
    ratio_stat_log = []
    collected_frames = 0

    pbar_cmp = tqdm(
        total=comparison_n_iters,
        desc=f"{algorithm} seed={seed}",
        leave=False,
    )

    for batch in collector_cmp:
        current_frames = batch.numel()
        collected_frames += current_frames

        _expand_agent_done_flags(batch, comparison_env.reward_key)

        with torch.no_grad():
            gae_cmp(
                batch,
                params=ppo_loss.critic_network_params,
                target_params=ppo_loss.target_critic_network_params,
            )

        # Capacity equals exactly one on-policy batch, so extending replaces
        # the previous batch. SamplerWithoutReplacement gives a new
        # permutation when each epoch is exhausted.
        replay_cmp.extend(batch.reshape(-1))

        last_critic_loss = float("nan")
        last_grad_norm = float("nan")
        last_entropy_loss = float("nan")
        last_kl = float("nan")
        last_ratio_stat = float("nan")

        for _ in range(comparison_ppo_epochs):
            for _ in range(
                comparison_frames_per_batch // comparison_minibatch_size
            ):
                subdata = replay_cmp.sample()
                loss_vals = ppo_loss(subdata)

                policy_loss = loss_vals["loss_objective"]
                critic_loss = loss_vals["loss_critic"]
                entropy_loss = loss_vals["loss_entropy"]
                total_loss = policy_loss + critic_loss + entropy_loss

                ppo_optim.zero_grad(set_to_none=True)
                total_loss.backward()
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    ppo_loss.parameters(), max_grad_norm
                )
                ppo_optim.step()
                ppo_optim.zero_grad(set_to_none=True)

                last_critic_loss = critic_loss.detach().item()
                last_grad_norm = float(grad_norm.detach().cpu())
                last_entropy_loss = entropy_loss.detach().item()

                loss_keys = set(loss_vals.keys())
                if "kl_approx" in loss_keys:
                    last_kl = loss_vals["kl_approx"].detach().mean().item()
                if "clip_fraction" in loss_keys:
                    last_ratio_stat = (
                        loss_vals["clip_fraction"].detach().mean().item()
                    )
                elif "ESS" in loss_keys:
                    last_ratio_stat = loss_vals["ESS"].detach().mean().item()

        collector_cmp.update_policy_weights_()

        batch_return = _batch_team_return(batch)
        frames_log.append(collected_frames)
        return_log.append(batch_return)
        critic_loss_log.append(last_critic_loss)
        grad_norm_log.append(last_grad_norm)
        entropy_loss_log.append(last_entropy_loss)
        kl_log.append(last_kl)
        ratio_stat_log.append(last_ratio_stat)

        pbar_cmp.set_postfix(
            frames=collected_frames,
            team_return=f"{batch_return:.2f}",
            refresh=False,
        )
        pbar_cmp.update()

    pbar_cmp.close()

    # Deterministic interaction mode removes stochastic sampling at final
    # evaluation while preserving the decentralized actor inputs.
    evaluation = _evaluate_final_policy(actor, "PPO", seed)
    saved_state = _cpu_state_dict(actor) if save_policy else None

    result = {
        "algorithm": algorithm,
        "seed": seed,
        "frames": frames_log,
        "team_return": return_log,
        "critic_loss": critic_loss_log,
        "grad_norm": grad_norm_log,
        "entropy_loss": entropy_loss_log,
        "kl_approx": kl_log,
        "ratio_stat": ratio_stat_log,
        "evaluation": evaluation,
    }

    try:
        collector_cmp.shutdown()
    except Exception:
        pass
    try:
        comparison_env.close()
    except Exception:
        pass

    del replay_cmp, ppo_loss, critic, actor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result, saved_state


In [ ]:
# Run all four algorithms with the same interaction budget and seeds.
# Expected runtime is much longer than a single MAPPO run because this executes
# 4 algorithms x 3 seeds. On a Kaggle GPU, the default is intended as the
# assignment-quality run rather than a quick smoke test.

comparison_results = {name: [] for name in comparison_algorithms}
representative_policy_states = {}

for seed in comparison_seeds:
    maddpg_result, maddpg_state = _train_ddpg_comparison(
        seed, centralised=True, save_policy=(seed == comparison_seeds[0])
    )
    comparison_results["MADDPG"].append(maddpg_result)
    if maddpg_state is not None:
        representative_policy_states["MADDPG"] = maddpg_state

    iddpg_result, _ = _train_ddpg_comparison(
        seed, centralised=False, save_policy=False
    )
    comparison_results["IDDPG"].append(iddpg_result)

    mappo_result, mappo_state = _train_ppo_comparison(
        seed, centralised=True, save_policy=(seed == comparison_seeds[0])
    )
    comparison_results["MAPPO"].append(mappo_result)
    if mappo_state is not None:
        representative_policy_states["MAPPO"] = mappo_state

    ippo_result, _ = _train_ppo_comparison(
        seed, centralised=False, save_policy=False
    )
    comparison_results["IPPO"].append(ippo_result)

print("Controlled CTDE runs completed.")


In [ ]:
# Learning curves against ENVIRONMENT FRAMES with mean +/- 1 standard error.
plt.figure(figsize=(10, 6))

learning_curve_summary = {}

for algorithm in comparison_algorithms:
    runs = comparison_results[algorithm]
    min_len = min(len(run["frames"]) for run in runs)
    frames = np.asarray(runs[0]["frames"][:min_len], dtype=float)
    curves = np.asarray(
        [run["team_return"][:min_len] for run in runs],
        dtype=float,
    )

    mean_curve = np.nanmean(curves, axis=0)
    if curves.shape[0] > 1:
        sem_curve = np.nanstd(curves, axis=0, ddof=1) / np.sqrt(curves.shape[0])
    else:
        sem_curve = np.zeros_like(mean_curve)

    learning_curve_summary[algorithm] = {
        "frames": frames,
        "mean": mean_curve,
        "sem": sem_curve,
    }

    plt.plot(frames, mean_curve, label=algorithm)
    plt.fill_between(
        frames,
        mean_curve - sem_curve,
        mean_curve + sem_curve,
        alpha=0.2,
    )

plt.xlabel("Environment frames")
plt.ylabel("Mean team return")
plt.title(
    f"Controlled CTDE comparison: mean +/- SEM over {len(comparison_seeds)} seeds"
)
plt.legend()
plt.grid(alpha=0.25)
plt.show()


# Final-policy evaluation: no DDPG training noise and deterministic PPO actions.
evaluation_rows = []
for algorithm in comparison_algorithms:
    for run in comparison_results[algorithm]:
        evaluation = run["evaluation"]
        row = {
            "algorithm": algorithm,
            "seed": run["seed"],
            "train_frames": comparison_total_frames,
            "eval_episodes": evaluation["n_eval_episodes"],
            "team_return_mean": evaluation["team_return_mean"],
            "team_return_std_within_seed": evaluation["team_return_std"],
        }
        for agent_idx, agent_return in enumerate(evaluation["per_agent_return"]):
            row[f"agent_{agent_idx}_return"] = agent_return
        evaluation_rows.append(row)

evaluation_df = pd.DataFrame(evaluation_rows)
print("Per-seed final evaluation:")
ipy_display(evaluation_df.round(3))

summary_rows = []
for algorithm in comparison_algorithms:
    sub = evaluation_df[evaluation_df["algorithm"] == algorithm]
    row = {
        "algorithm": algorithm,
        "team_return_mean_across_seeds": sub["team_return_mean"].mean(),
        "team_return_std_across_seeds": sub["team_return_mean"].std(ddof=1),
    }
    for agent_idx in range(n_agents):
        col = f"agent_{agent_idx}_return"
        row[f"agent_{agent_idx}_mean_across_seeds"] = sub[col].mean()
    summary_rows.append(row)

evaluation_summary_df = pd.DataFrame(summary_rows)
print("Final evaluation summary across seeds:")
ipy_display(evaluation_summary_df.round(3))


In [ ]:
# Representative deterministic evaluation episode for each algorithm family.
# We render the centralized member (MADDPG and MAPPO) from the first seed.
from PIL import Image as PILImage
from IPython.display import Image as IPImage, display as ipy_display

try:
    import pyvirtualdisplay
    _comparison_display = pyvirtualdisplay.Display(
        visible=False, size=(1400, 900)
    )
    _comparison_display.start()
except Exception:
    _comparison_display = None


def _render_representative_episode(algorithm, policy_state):
    render_seed = comparison_seeds[0] + 20_000
    render_env = _make_comparison_env(render_seed, num_envs=1)

    if algorithm == "MADDPG":
        render_policy = _build_ddpg_actor(render_env)
    elif algorithm == "MAPPO":
        render_policy = _build_ppo_actor(render_env)
    else:
        raise ValueError("Representative rendering is defined for MADDPG or MAPPO.")

    render_policy.load_state_dict(policy_state)
    render_policy.eval()

    frames = []

    def _render_callback(current_env, td):
        frames.append(
            PILImage.fromarray(current_env.render(mode="rgb_array"))
        )

    with torch.no_grad(), set_exploration_type(ExplorationType.DETERMINISTIC):
        rollout = render_env.rollout(
            max_steps=comparison_max_steps,
            policy=render_policy,
            callback=_render_callback,
            auto_cast_to_device=True,
            break_when_any_done=False,
        )

    gif_name = f"ctde_{algorithm.lower()}_seed{comparison_seeds[0]}.gif"
    if frames:
        frames[0].save(
            gif_name,
            save_all=True,
            append_images=frames[1:],
            duration=40,
            loop=0,
        )

    completed = _completed_per_agent_returns(rollout)
    representative_per_agent = completed.mean(dim=0).detach().cpu().numpy()
    representative_team = float(representative_per_agent.sum())

    try:
        collision_signal = rollout.get(
            ("next", "agents", "info", "agent_collisions")
        )
        collision_entries = int(
            (collision_signal.detach().abs() > 1e-8).sum().cpu().item()
        )
    except Exception:
        collision_entries = None

    try:
        render_env.close()
    except Exception:
        pass

    return {
        "gif": gif_name,
        "team_return": representative_team,
        "per_agent_return": representative_per_agent.tolist(),
        "collision_entries": collision_entries,
    }


representative_stats = {}
for algorithm in ["MADDPG", "MAPPO"]:
    representative_stats[algorithm] = _render_representative_episode(
        algorithm,
        representative_policy_states[algorithm],
    )

for algorithm, stats in representative_stats.items():
    print(
        f"{algorithm}: team return={stats['team_return']:.3f}, "
        f"per-agent returns={np.round(stats['per_agent_return'], 3).tolist()}, "
        f"non-zero collision entries={stats['collision_entries']}"
    )
    if stats["collision_entries"] == 0:
        print(
            "  Coordination observation: this representative rollout recorded "
            "no non-zero agent-collision entries; inspect the GIF to confirm "
            "whether agents separate cleanly and reach distinct goals."
        )
    elif stats["collision_entries"] is not None:
        print(
            "  Coordination observation: this representative rollout recorded "
            f"{stats['collision_entries']} non-zero agent-collision entries, "
            "which is evidence of imperfect collision avoidance; inspect the GIF "
            "for where the coordination failure occurs."
        )
    else:
        print(
            "  Coordination observation: collision info was unavailable; use "
            "the rendered GIF and per-agent returns to describe a visible "
            "coordination success or failure."
        )

    ipy_display(IPImage(filename=stats["gif"]))


# Answers

## Question 1: Stability versus Sample Efficiency

### 1. Why is MAPPO usually more stable early in training?

MAPPO uses **stochastic exploration, PPO clipping, and GAE**, which make updates smoother. MADDPG uses a bootstrapped critic and replay data from older policies, so critic errors and multi-agent non-stationarity can make training unstable.

### 2. Why can MADDPG be more sample efficient?

MADDPG can **reuse the same transitions many times** from its replay buffer, so it can perform more gradient updates without collecting new environment data. MAPPO needs mostly fresh on-policy data. The downside is that old replay data can become stale.

### 3. Example of instability from our run

MADDPG's return dropped from about **-5.6 to -10.7** early in training while critic loss was noisy. The exploration scale stayed fixed at **0.5**, so the drop was likely caused by unstable critic estimates and replay from changing agent policies.

---

## Question 2: Centralization, Parameter Sharing, and Credit Assignment

### 1. MADDPG vs IDDPG and MAPPO vs IPPO

* **MADDPG:** decentralized actor, centralized critic using all agents' observations and actions.
* **IDDPG:** same actor, but critic uses only local observation and action.
* **MAPPO:** decentralized actor, centralized value critic using all agents' observations.
* **IPPO:** same actor, but value critic uses only local observation.

In all cases, actors remain shared and decentralized, so execution-time information does not change.

### 2. Problems with a decentralized MAPPO critic

A local critic cannot see the other agents, which can cause:

* worse partial observability,
* harder credit assignment,
* higher advantage variance,
* weaker coordination between similar agents.

In our experiment, IPPO still performed well, but MAPPO had a slightly higher final return.

### 3. What did our ablation show?

Final team returns were:

| Algorithm | Mean Return |
| --------- | ----------: |
| MADDPG    |      -44.28 |
| IDDPG     |      -61.73 |
| MAPPO     |        9.07 |
| IPPO      |        8.37 |

Centralization helped **MADDPG clearly** and helped **MAPPO slightly**. MADDPG also had lower variance than IDDPG.

However, this does not prove CTDE alone caused the difference because centralized critics have different input sizes, and we used only three seeds.

---

## Original Notebook Questions

**1. Why is MAPPO faster and more stable?**
Because PPO clipping, stochastic exploration, and GAE give smoother updates.

**2. Why is MADDPG more sample efficient?**
Because replay allows the same environment experience to be reused many times.

**3. What happens with a decentralized MAPPO critic?**
It loses information about other agents, making credit assignment and coordination harder.

**4. IDDPG vs MADDPG and IPPO vs MAPPO?**
MADDPG/MAPPO use centralized critics; IDDPG/IPPO use decentralized critics. The actors stay decentralized in all cases.